# A/B Test Design & Simulation: Right Customer × Right Agent

**Question:** Does routing customers to a *matched* agent (segment × agent score) increase sales conversion compared with the current routing?

| Item | Choice |
|---|---|
| Unit of randomization | Customer (TCKN), not call: one customer must always stay in the same group |
| Control (A) | Current dialer routing |
| Treatment (B) | Customer routed to the best-matched agent |
| Primary metric | Policy conversion within 14 days of first call |
| Guardrails | Reach rate, avg. talk time, calls per agent |
| Test | Two-proportion z-test, α = 0.05, power = 0.80 |

> Replace the assumptions in the next cell with real numbers from `RPT_GUNLUK_URETIM` and the call history before running.

In [1]:
import numpy as np
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep

rng = np.random.default_rng(42)

# --- Assumptions (replace with real figures) ---
BASELINE_CR   = 0.040   # current conversion rate per called customer
MDE_RELATIVE  = 0.10    # smallest uplift worth detecting: +10% relative
DAILY_CUSTOMERS = 6000  # customers called per day (both groups together)
ALPHA, POWER = 0.05, 0.80

## 1. Sample size & duration (power analysis)

In [2]:
target_cr = BASELINE_CR * (1 + MDE_RELATIVE)
effect = proportion_effectsize(target_cr, BASELINE_CR)
n_per_group = int(np.ceil(NormalIndPower().solve_power(effect, alpha=ALPHA, power=POWER, ratio=1)))
days = int(np.ceil(2 * n_per_group / DAILY_CUSTOMERS))

print(f"Baseline {BASELINE_CR:.2%} -> target {target_cr:.2%}")
print(f"Required customers per group: {n_per_group:,}")
print(f"Estimated duration: {days} days (round up to full weeks: {int(np.ceil(days/7))*7} days)")

Baseline 4.00% -> target 4.40%
Required customers per group: 39,455
Estimated duration: 14 days (round up to full weeks: 14 days)


**Why round up to full weeks?** Call-center behaviour differs by weekday; a test that covers only part of a week is biased.

How MDE drives sample size:

In [3]:
for mde in [0.05, 0.10, 0.15, 0.20]:
    t = BASELINE_CR * (1 + mde)
    n = NormalIndPower().solve_power(proportion_effectsize(t, BASELINE_CR), alpha=ALPHA, power=POWER)
    print(f"MDE +{mde:.0%}: {int(np.ceil(n)):>8,} per group  ~{int(np.ceil(2*n/DAILY_CUSTOMERS))} days")

MDE +5%:  154,283 per group  ~52 days
MDE +10%:   39,455 per group  ~14 days
MDE +15%:   17,923 per group  ~6 days
MDE +20%:   10,297 per group  ~4 days


## 2. Analysis function

In [4]:
def analyze(conv_a, n_a, conv_b, n_b, alpha=ALPHA):
    z, p = proportions_ztest([conv_b, conv_a], [n_b, n_a])
    lo, hi = confint_proportions_2indep(conv_b, n_b, conv_a, n_a, compare="diff", alpha=alpha)
    cr_a, cr_b = conv_a / n_a, conv_b / n_b
    return {"cr_a": cr_a, "cr_b": cr_b, "uplift_rel": cr_b / cr_a - 1,
            "diff_ci": (lo, hi), "p_value": p, "significant": p < alpha}

## 3. Simulate one experiment with a true +10% effect

In [5]:
n = n_per_group
conv_a = rng.binomial(n, BASELINE_CR)
conv_b = rng.binomial(n, target_cr)
r = analyze(conv_a, n, conv_b, n)
print(f"A: {r['cr_a']:.3%}  B: {r['cr_b']:.3%}  uplift: {r['uplift_rel']:+.1%}")
print(f"95% CI of difference: [{r['diff_ci'][0]:+.3%}, {r['diff_ci'][1]:+.3%}]  p = {r['p_value']:.4f}")

A: 3.825%  B: 4.339%  uplift: +13.5%
95% CI of difference: [+0.238%, +0.791%]  p = 0.0003


## 4. Sanity checks via simulation
**A/A test:** no real difference → we should see ~5% false positives.
**Power check:** with the real +10% effect → we should detect it ~80% of the time.

In [6]:
def run_many(cr_a, cr_b, n, sims=2000):
    hits = 0
    for _ in range(sims):
        hits += analyze(rng.binomial(n, cr_a), n, rng.binomial(n, cr_b), n)["significant"]
    return hits / sims

print(f"A/A false positive rate: {run_many(BASELINE_CR, BASELINE_CR, n):.1%}  (expected ~5%)")
print(f"Empirical power:         {run_many(BASELINE_CR, target_cr, n):.1%}  (expected ~80%)")

A/A false positive rate: 5.9%  (expected ~5%)


Empirical power:         79.5%  (expected ~80%)


## 5. Why we must not peek
Checking the p-value every day and stopping at the first p < 0.05 inflates false positives, even when there is **no** real effect.

In [7]:
def peeking_fpr(cr, n_total, days, sims=1000):
    daily = n_total // days
    fp = 0
    for _ in range(sims):
        a = rng.binomial(1, cr, n_total); b = rng.binomial(1, cr, n_total)
        for d in range(1, days + 1):
            k = d * daily
            if analyze(a[:k].sum(), k, b[:k].sum(), k)["significant"]:
                fp += 1; break
    return fp / sims

print(f"False positive rate with daily peeking: {peeking_fpr(BASELINE_CR, n, days):.1%}  (vs 5% without peeking)")

False positive rate with daily peeking: 23.3%  (vs 5% without peeking)


## 6. Decision rules (agree on these *before* launch)
1. Run for the full planned duration; no early stopping on the primary metric.
2. Ship if the primary metric is significant **and** no guardrail gets significantly worse.
3. Segment-level results (e.g., per value segment) are exploratory unless pre-registered.
4. Afterwards, keep the winner as *champion* and test new matching logic as *challenger* on a small share of traffic.

**Risk to note:** agents serve both groups, so treatment can spill over into control (interference). If this becomes a problem, randomize by agent team or by day instead of by customer.